In [ ]:
"""Module 1: Data loading helpers for ticket routing challenge."""

from pathlib import Path
from typing import Iterable

import pandas as pd


REQUIRED_COLUMNS = {"ticket_id", "message", "category"}


def load_ticket_data(path: Path | str) -> pd.DataFrame:
    """
    Load the ticket dataset for the routing challenge.

    TODO:
    - Read the CSV from the provided path (raise `FileNotFoundError` if missing).
    - Validate required columns using `ensure_required_columns`.
    - Drop records missing `message` or `category` and normalize whitespace.
    - Raise `ValueError` if the resulting DataFrame is empty.
    """

    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {path}")

    df = ensure_required_columns(df, REQUIRED_COLUMNS)
    df["message"] = df["message"].str.strip()
    df["category"] = df["category"].str.strip()

    df.replace("", pd.NA, inplace=True)
    df = df.dropna(subset=["message", "category"])

    if df.empty:
        raise ValueError("No valid records found.")

    return df


def ensure_required_columns(df: pd.DataFrame, columns: Iterable[str]) -> pd.DataFrame:
    """
    Ensure the DataFrame contains a required set of columns.

    TODO:
    - Identify missing columns and raise `ValueError` if any are absent.
    - Return the original DataFrame when validation succeeds to enable chaining.
    """

    missing_columns = [col for col in columns if col not in df.columns]

    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}")

    return df


result = load_ticket_data(path="data.csv")
result.head()

,ticket_id,message,category
0,1,Password reset not working. Cannot log into my...,authentication
1,2,Charged twice for the premium plan invoice las...,billing
2,3,App keeps crashing whenever I upload a CSV file.,technical
3,4,How do I change the email associated with my a...,account
4,5,Need documentation for integrating your API wi...,product


In [ ]:
"""Module 2: Ticket routing model."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, List

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB


@dataclass
class PredictionResult:
    """Structured response for ticket routing predictions."""

    queue: str
    confidence: float


class TicketRouter:
    """Trainable text classification pipeline with confidence fallback."""

    def __init__(self, *, threshold: float = 0.55) -> None:
        """
        Initialize the router.

        Args:
            threshold: Minimum confidence required to auto-route tickets.
        """

        self.threshold = threshold
        self.model = MultinomialNB()
        self.vectorizer = TfidfVectorizer()

    def fit(self, texts: Iterable[str], labels: Iterable[str]) -> None:
        """Train the router on past ticket messages and categories."""

        X = self.vectorizer.fit_transform(texts)
        self.model.fit(X, labels)

    def predict_with_confidence(self, texts: Iterable[str]) -> List[PredictionResult]:
        """Predict ticket queues with associated confidence scores."""

        # Check model not fitted
        if not hasattr(self.model, "classes_"):
            raise RuntimeError("Model has not been fitted. Call `fit` first.")

        X = self.vectorizer.transform(texts)
        probabilities = self.model.predict_proba(X)
        labels = self.model.classes_

        results: List[PredictionResult] = []
        for prob in probabilities:
            max_index = prob.argmax()
            label = labels[max_index]
            confidence = prob[max_index]
            result = self._format_prediction(label, confidence)
            results.append(result)

        return results

    def _format_prediction(self, label: str, probability: float) -> PredictionResult:
        """Apply thresholding logic to produce prediction results."""

        if probability >= self.threshold:
            return PredictionResult(queue=label, confidence=probability)
        else:
            return PredictionResult(queue="manual review", confidence=probability)


ticket_router = TicketRouter()
ticket_router.fit(
    texts=[
        "I need help with my order",
        "Technical issue with the product",
        "Billing question about my invoice",
    ],
    labels=[
        "support",
        "technical",
        "billing",
    ]
)

ticket_router.predict_with_confidence([
    "I want to know about my order status",
    "There is a bug in the application",
    "I need a copy of my invoice",
])

[PredictionResult(queue='manual review', confidence=np.float64(0.37677531907283107)),
 PredictionResult(queue='manual review', confidence=np.float64(0.4231318012738312)),
 PredictionResult(queue='manual review', confidence=np.float64(0.37677531907283107))]

In [22]:
"""Module 3: Rule-based entity extraction utilities."""

from __future__ import annotations

from dataclasses import dataclass
import re
from typing import Dict, List, Tuple


@dataclass
class Entity:
    """Structured entity match."""

    text: str
    start: int
    end: int


class EntityExtractor:
    """Regex-based helper for extracting customer entities."""

    def __init__(self) -> None:
        self.patterns: Dict[str, List[re.Pattern[str]]] = {
            "PERSON": [
                re.compile(r"\b[A-Z][a-z]+ [A-Z][a-z]+\b"),
                re.compile(r"\bMr\. [A-Z][a-z]+\b"),
                re.compile(r"\bMs\. [A-Z][a-z]+\b"),
            ],
            "EMAIL": [
                re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
            ],
            "PHONE": [
                re.compile(r"\(\d{3}\)\s?\d{3}-\d{4}(?=\b|\D)"),
                re.compile(r"\d{3}-\d{3}-\d{4}(?=\b|\D)"),
            ],
            "MONEY": [
                re.compile(r"\$[\d,]+\.?\d*\b"),
            ],
        }

    def extract(self, text: str) -> Dict[str, List[Entity]]:
        """Extract entities from unstructured text."""
        entities: Dict[str, List[Entity]] = {key: [] for key in self.patterns.keys()}

        for key, patterns in self.patterns.items():
            for pattern in patterns:
                for match in pattern.finditer(text):
                    entity = Entity(text=match.group(), start=match.start(), end=match.end())
                    entities[key].append(entity)

        return entities

    def annotate(self, text: str) -> str:
        """Return text with inline entity annotations."""
        entities = self.extract(text)
        for key, entity_list in entities.items():
            for entity in entity_list:
                text = text.replace(entity.text, f"[{key}|{entity.text}]")
        return text


entity_extractor = EntityExtractor()
entity_extractor.annotate("Contact Sarah Johnson at sarah@example.com about $129.50 invoice.")

'[PERSON|Contact Sarah] Johnson at [EMAIL|sarah@example.com] about [MONEY|$129.50] invoice.'